# [2.2.1] - DQN (exercises)

> **ARENA [Streamlit Page](https://arena-chapter2-rl.streamlit.app/21_[2.2.1]_DQN)**
>
> **Colab: [exercises](https://colab.research.google.com/github/ARENA-education/ARENA_materials/blob/main/chapter2_rl/exercises/part21_dqn/2.2.1_DQN_exercises.ipynb?t=20260929) | [solutions](https://colab.research.google.com/github/ARENA-education/ARENA_materials/blob/main/chapter2_rl/exercises/part21_dqn/2.2.1_DQN_solutions.ipynb?t=20260929)**

Please send any problems / bugs on the `#errata` channel in the [Slack group](https://info-arena.github.io/ARENA_img/slack.html), and ask any questions on the dedicated channels for this chapter of material.

You can collapse each section so only the headers are visible, by clicking the arrow symbol on the left hand side of the markdown header cells.

Links to all other chapters: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/headers/header-22.png" width="350">

# Introduction

In this section, you'll implement Deep Q-Learning, often referred to as DQN for "Deep Q-Network".
Presented in [Playing Atari with Deep Reinforcement Learning](https://www.cs.toronto.edu/~vmnih/docs/dqn.pdf),
this was one of the first successful applications of Deep Learning to Reinforcement Learning.

## Content & Learning Objectives


### 1️⃣ DQN

In this section, you'll implement Deep Q-Learning, often referred to as DQN for "Deep Q-Network". This was used in a landmark paper [Playing Atari with Deep Reinforcement Learning](https://www.cs.toronto.edu/~vmnih/docs/dqn.pdf).

You'll apply the technique of DQN to master the famous CartPole environment (below), and then (if you have time) move on to harder challenges like Acrobot and MountainCar.

> ##### Learning Objectives
>
> - Understand the DQN algorithm
> - Learn more about RL debugging, and build probe environments to debug your agents
> - Create a replay buffer to store environment transitions
> - Implement DQN using PyTorch, on the CartPole environment

## Setup (don't read, just run!)

In [16]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter2_rl"
repo = "ARENA_materials"
branch = "main"

# Install dependencies
try:
    import jaxtyping
except:
    %pip install "wandb>=0.18" einops "gymnasium[atari,other]>=1.2,<2" "ale-py>=0.10" pygame jaxtyping torchinfo git+https://github.com/ARENA-education/eindex.git

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/ARENA-education/ARENA_materials/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/{repo}-{branch}


if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [17]:
import sys
import time
import warnings
from collections import namedtuple
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable, Optional

import gymnasium as gym
import numpy as np
import torch as t
import torch.nn.functional as F
import wandb
from eindex import eindex
from gymnasium.spaces import Box, Discrete
from jaxtyping import Bool, Float, Int
from torch import Tensor, nn
from torchinfo import summary
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")


# Make sure exercises are in the path
chapter = "chapter2_rl"
section = "part21_dqn"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

from gpu_env import CartPole, MountainCar
from gpu_probe import Probe1, Probe2, Probe3, Probe4, Probe5
from rl_utils import ENVS, AtariEnvs, LiveVideo, log_greedy_rollout_video, log_grid_video, make_envs
import part21_dqn.tests as tests
from part1_intro_to_rl.utils import set_global_seeds
from plotly_utils import line, plot_cartpole_obs_and_dones

# The networks in this section are tiny, and if you train on the CPU with
# PyTorch's default of one thread per core, threads spend longer
# coordinating than computing. Adjust if needed.
t.set_num_threads(min(4, t.get_num_threads()))

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")


MAIN = __name__ == "__main__"

# 1️⃣ Q-Network & Replay Buffer

> ##### Learning Objectives
>
> - Understand the DQN algorithm
> - Learn more about RL debugging, and build probe environments to debug your agents
> - Create a replay buffer to store environment transitions
> - Implement DQN using PyTorch, on the CartPole environment

In this section, you'll implement Deep Q-Learning, often referred to as DQN for "Deep Q-Network". This was used in a landmark paper [Playing Atari with Deep Reinforcement Learning](https://www.cs.toronto.edu/~vmnih/docs/dqn.pdf).

At the time, the paper was very exciting: The agent would play the game by only looking at the same screen pixel data that a human player would be looking at, rather than a description of where the enemies in the game world are. The idea that convolutional neural networks could look at Atari game pixels and "see" gameplay-relevant features like a Space Invader was new and noteworthy. In 2022, we take for granted that convnets work, so we're going to focus on the RL aspect solely, and not the vision component.

For a lecture on the material in this section, which provides some high-level understanding before you dive into the exercises, watch the video below:

<iframe width="540" height="304" src="https://www.youtube.com/embed/_4AHUdpkarM?start=1329" frameborder="0" referrerpolicy="strict-origin-when-cross-origin" allow="accelerometer; autoplay; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>

## Optional Readings

* [Deep Q Networks Explained](https://www.lesswrong.com/posts/kyvCNgx9oAwJCuevo/deep-q-networks-explained) (25 minutes)
    * A high-level distillation as to how DQN works.
    * Read sections 1-4 (further sections optional).
* [Andy Jones - Debugging RL, Without the Agonizing Pain](https://andyljones.com/posts/rl-debugging.html) (10 minutes)
    * Useful tips for debugging your code when it's not working.
    * Read up to (not including) the Common Fixes section. Also read the Practical Advice section, up to and including "Use probe agents". The rest of the post is optional, and you're recommended to come back to it near the end if you're stuck.
    * The "probe environments" (a collection of simple environments of increasing complexity) section will be our first line of defense against bugs, you'll implement these in exercises below.

### Interesting Resources (not required reading)

- [An Outsider's Tour of Reinforcement Learning](https://archives.argmin.net/2018/06/25/outsider-rl/) - comparison of RL techniques with the engineering discipline of control theory.
- [Towards Characterizing Divergence in Deep Q-Learning](https://arxiv.org/pdf/1903.08894.pdf) - analysis of what causes learning to diverge
- [Divergence in Deep Q-Learning: Tips and Tricks](https://amanhussain.com/post/divergence-deep-q-learning/) - includes some plots of average returns for comparison
- [Deep RL Bootcamp](https://sites.google.com/view/deep-rl-bootcamp/lectures) - 2017 bootcamp with video and slides. Good if you like videos.
- [DQN debugging using OpenAI gym Cartpole](https://adgefficiency.com/dqn-debugging/) - random dude's adventures in trying to get it to work.
- [CleanRL DQN](https://github.com/vwxyzjn/cleanrl) - single file implementations of RL algorithms. Your starter code today is based on this; try not to spoiler yourself by looking at the solutions too early!
- [Deep Reinforcement Learning Doesn't Work Yet](https://www.alexirpan.com/2018/02/14/rl-hard.html) - 2018 article describing difficulties preventing industrial adoption of RL.
- [Deep Reinforcement Learning Works - Now What?](https://tesslerc.github.io/posts/drl_works_now_what/) - 2020 response to the previous article highlighting recent progress.
- [Seed RL](https://github.com/google-research/seed_rl) - example of distributed RL using Docker and GCP.

### Conceptual overview of DQN

DQN is the natural extension of Q-Learning into the domain of deep learning. The main difference is that, instead of a table to store all the Q-values for each state-action pair, we train a neural network to learn this function for us. The usual implementation (which we'll use here) is for the Q-network to take the state as input, and output a vector of optimal Q-values for each action, i.e. we're learning the function:

$$
s \to (Q^*(s, a_1), ..., Q^*(s, a_n))
$$

Below is an algorithm showing the conceptual overview of DQN. We cycle through the following process:

* Generate a batch of experiences using our current policy, by **epsilon-greedy sampling** (i.e. we mostly take the action with the highest Q-value, but occasionally take a random action to encourage exploration). Store these experiences in the **replay buffer**.
* Use these values to calculate a **TD (temporal difference) error**, and update our network.
    * To increase stability, we also have a **target network** we use for the "next step" part of the TD error. This is a lagged copy of the Q-network (i.e. we update our Q-network via gradient descent, and then every so often we copy the Q-network weights over to our target network).
* Repeat this until convergence.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/ppo-alg-conceptual-2.png" width="750">

### Fast Feedback Loops

We want to have faster feedback loops, and learning from Atari pixels doesn't achieve that. It might take 15 minutes per training run to get an agent to do well on Breakout, and that's if your implementation is relatively optimized. Even waiting 5 minutes to learn Pong from pixels is going to limit your ability to iterate, compared to using environments that are as simple as possible.

### CartPole

The classic environment "CartPole-v1" is simple to understand, yet hard enough for a RL agent to be interesting, by the end of the day your agent will be able to do this and more! (Click to watch!)


[![CartPole](https://img.youtube.com/vi/46wjA6dqxOM/0.jpg)](https://www.youtube.com/watch?v=46wjA6dqxOM "CartPole")

If you'd like to try the CartPole environment yourself, [click here to open the simulation](https://davidquarel.github.io/cartpole) in a new tab.
* Use Left/Right arrow keys to move the cart,
* R to reset,
* Q to quit.
* Use F/S to make the simulation faster/slower.

Unlike the real CartPole environment, this simulation will not terminate the episode if the pole falls over.
We've also cheated here and added a hidden third no-op action, such that if no button is pressed, no force is applied to the cart.
This makes the simulation a bit easier for you as the human to play.
The real cartpole environment doesn't act like this: the agent must choose to push the cart either left or right on each timestep.

The description of the task is [here](https://gymnasium.farama.org/environments/classic_control/cart_pole/). Note that unlike the previous environments, the observation here is now continuous. You can see the source for CartPole [here](https://github.com/Farama-Foundation/Gymnasium/blob/main/gymnasium/envs/classic_control/cartpole.py); don't worry about the implementation but do read the documentation to understand the format of the actions and observations.

The simple physics involved would be very easy for a model-based algorithm to fit, (this is a common assignment in control theory using [proportional-integral-derivative](https://en.wikipedia.org/wiki/PID_controller) (PID) controllers) but today we're doing it model-free: your agent has no idea that these observations represent positions or velocities, and it has no idea what the laws of physics are. The network has to learn in which direction to bump the cart in response to the current state of the world.

Each environment can have different versions registered to it. By consulting [the Gym source](https://github.com/Farama-Foundation/Gymnasium/blob/main/gymnasium/envs/__init__.py) you can see that CartPole-v0 and CartPole-v1 are the same environment, except that v1 has longer episodes. Again, a minor change like this can affect what algorithms score well; an agent might consistently survive for 200 steps in an unstable fashion that means it would fall over if run for 500 steps.

We would like to run many copies of this environment in parallel to collect trajectories as fast as possible.
The older way of doing this involved using [AsyncVectorEnv](https://gymnasium.farama.org/api/vector/async_vector_env/)
provided by `gymnasium`, which would spawn `num_envs` many processes, one for each environment.
This is nice, as it works for an arbitrary environment without any additional effort, but
it doesn't really scale that well. Each process is independent, and each carries their own Python interpreter.
Given that the update rule for the environment itself is trivial,
this overhead is significant. A better approach would be to modify the environment to do the parallelism internally,
spawn a pool of workers and then have threads compute the update rule, but what we do instead is we
port the entire environment dynamics to PyTorch (see `chapter2_rl/exercises/gpu_env.py`), where
the observations are natively stored as a tensor, and all environments are run in lock-step.

This means
* we don't need to constantly convert between numpy and torch tensors, simplifying code
* we can run large numbers of environments in parallel (~thousands of environments for ~millions of
environmental steps per second)
* for toy examples like this where neither the GPU nor the CPU is doing a lot of work, the main bottleneck
can just be moving data back and forth between the two.

However, it does add some additional complexity in that it forces all environments to be run for the same
number of timesteps, which can cause trouble if some trajectories terminate faster than others.
We handle this by allowing the environment to automatically reset upon termination, meaning some
rollouts may have more or fewer disjoint episodes, or that an episode may get cut-off halfway.

In [18]:
envs = CartPole(num_envs=4, device=device)

print(envs.single_action_space)  # 2 actions: left and right
print(envs.single_observation_space)  # Box(4): each observation can take a continuous range of values

obs, infos = envs.reset()
print(obs.shape, obs.device)  # (num_envs, 4): one observation per environment, as a tensor on our device

Discrete(2)
Box([-4.8000002e+00 -3.4028235e+38 -4.1887903e-01 -3.4028235e+38], [4.8000002e+00 3.4028235e+38 4.1887903e-01 3.4028235e+38], (4,), float32)
torch.Size([4, 4]) cuda:0


### Outline of the Exercises

The exercises are roughly split into 4 sections:

1. Implement the Q-network that maps a state to an estimated value for each action.
2. Implement a replay buffer to store experiences $e_t = (s_t, a_t, r_{t+1}, d_{t+1}, s_{t+1})$.
3. Implement the policy which chooses actions based on the Q-network, plus epsilon greedy randomness to encourage exploration.
4. Piece everything together into a training loop and train your agent.

## The Q-Network

The Q-Network takes in an observation $s$ and outputs a vector $[Q^*(s, a^1), \ldots Q^*(s,a^n)]$ representing an estimate of the optimal Q-value for the given state $s$, and each possible action $\mathcal{A} = \{a^1, \ldots, a^n\}$. This replaces our Q-value table used in Q-learning.

For best results, the architecture of the Q-network can be customized to each particular problem. For example, [the architecture of OpenAI Five](https://cdn.openai.com/research-covers/openai-five/network-architecture.pdf) used to play DOTA 2 is pretty complex and involves LSTMs.

For learning from pixels, a simple convolutional network and some fully connected layers does quite well. Where we have already processed features here, it's even easier: an MLP of this size should be plenty large for any environment today.

Implement the Q-network using a standard MLP, constructed of alternating Linear and ReLU layers.
The size of the input will match the dimensionality of the observation space, and the size of the output will match the number of actions to choose from (associating a reward to each.)
The dimensions of the hidden_sizes are provided.

Here is a diagram of what our particular Q-Network will look like for CartPole (you can open it in a new tab if it's hard to see clearly):

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/mermaid-diagram-2024-11-28-200952.svg" style="background-color: #fffbe0; padding: 10px" width="160"><br>

<!--
flowchart TD
    A["Input (num_obs,)"] -> B["Linear(num_obs, 120)"]
    B -> C[ReLU]
    C -> D["Linear(120, 84)"]
    D -> E[ReLU]
    E -> F["Linear(84, num_actions)"]
    F -> G["Output (num_actions,)"]
{
  "theme": "default",
  "themeVariables": {
    "fontSize": "22px"
  }
}
-->

<details>
<summary>Question - why do we not include a ReLU at the end?</summary>

If you end with a ReLU, then your network can only predict 0 or positive Q-values. This will cause problems as soon as you encounter an environment with negative rewards, or you try to do some scaling of the rewards.

</details>

<details>
<summary>Question - since CartPole-v1 gives +1 reward on every timestep, why do you think the network doesn't just learn the constant +1 function regardless of observation?</summary>

The network is learning Q-values (the sum of all future expected discounted rewards from this state/action pair), not rewards. Correspondingly, once the agent has learned a good policy, the Q-value associated with state action pair (pole is slightly left of vertical, move cart left) should be large, as we would expect a long episode (and correspondingly lots of reward) by taking actions to help to balance the pole. Pairs like (cart near right boundary, move cart right) cause the episode to terminate, and as such the network will learn low Q-values.

</details>

### Exercise - implement `QNetwork`

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
>
> You should spend up to 10-15 minutes on this exercise.
> ```

Note - in this implementation we can assume that `obs_shape` is a tuple of length 1 (in the case of CartPole this will be `(4,)`), so you can treat it as just an integer value above, e.g. your first linear layer should be from `obs_shape[0]` to `120`.

In [19]:
class QNetwork(nn.Module):
    """
    For consistency with your tests, please wrap your modules in a `nn.Sequential` called `layers`.
    """

    layers: nn.Sequential

    def __init__(self, obs_shape: tuple[int], num_actions: int, hidden_sizes: list[int] = [120, 84]):
        super().__init__()
        assert len(obs_shape) == 1, "Expecting a single vector of observations"
        self.layers = nn.Sequential(
            nn.Linear(obs_shape[0], hidden_sizes[0]),
            nn.ReLU(),
            nn.Linear(hidden_sizes[0], hidden_sizes[1]),
            nn.ReLU(),
            nn.Linear(hidden_sizes[1], num_actions)

        )

    def forward(self, x: Tensor) -> Tensor:
        return self.layers(x)


net = QNetwork(obs_shape=(4,), num_actions=2)
n_params = sum((p.nelement() for p in net.parameters()))
assert isinstance(getattr(net, "layers", None), nn.Sequential)
print(net)
print(f"Total number of parameters: {n_params}")
print("You should manually verify network is Linear-ReLU-Linear-ReLU-Linear")
assert not isinstance(net.layers[-1], nn.ReLU)
assert n_params == 10934

QNetwork(
  (layers): Sequential(
    (0): Linear(in_features=4, out_features=120, bias=True)
    (1): ReLU()
    (2): Linear(in_features=120, out_features=84, bias=True)
    (3): ReLU()
    (4): Linear(in_features=84, out_features=2, bias=True)
  )
)
Total number of parameters: 10934
You should manually verify network is Linear-ReLU-Linear-ReLU-Linear


<details><summary>Solution</summary>

```python
class QNetwork(nn.Module):
    """
    For consistency with your tests, please wrap your modules in a `nn.Sequential` called `layers`.
    """

    layers: nn.Sequential

    def __init__(self, obs_shape: tuple[int], num_actions: int, hidden_sizes: list[int] = [120, 84]):
        super().__init__()
        assert len(obs_shape) == 1, "Expecting a single vector of observations"
        in_features_list = [obs_shape[0]] + hidden_sizes
        out_features_list = hidden_sizes + [num_actions]
        layers = []
        for i, (in_features, out_features) in enumerate(zip(in_features_list, out_features_list)):
            layers.append(nn.Linear(in_features, out_features))
            if i < len(in_features_list) - 1:
                layers.append(nn.ReLU())
        self.layers = nn.Sequential(*layers)

    def forward(self, x: Tensor) -> Tensor:
        return self.layers(x)
```
</details>

## Replay Buffer

The goal of DQN is to reduce the reinforcement learning problem to a supervised learning problem. In supervised learning, training examples should be drawn **identically and independently distributed (i.i.d.)** from some distribution, and we hope to generalize to future examples from that distribution. Obviously perfect i.i.d. sampling isn't attainable, but we can approximate this by filling a buffer of past experiences and sampling from it. Note that for very complex problems we may need a very large buffer, because we want the policy to get a representative sample of all the diverse scenarios that can happen in the environment. [OpenAI Five](https://cdn.openai.com/dota-2.pdf) used batch sizes of over 2 million experiences for Dota 2! However we'll be working with the fairly simple CartPole environment, and so we can get away with a much smaller buffer.

In RL, the distribution of experiences $e_t = (s_t, a_t, r_{t+1}, s_{t+1})$ to train from depends on the policy $\pi$ followed, which depends on the current state of the Q-value network, so DQN is always chasing a moving target. This is why the training loss curve isn't going to have a nice steady decrease like in supervised learning. We will extend experiences to $e_t = (s_t, a_t, r_{t+1}, s_{t+1}, d_{t+1})$. Here, $d_{t+1}$ is a boolean indicating that $s_{t+1}$ is a terminal observation, and that no further interaction happened beyond $s_{t+1}$ in the episode from which it was generated.

### Termination vs Truncation

Note that we take $d_{t+1}$ to be `terminated`, not `done = terminated | truncated`. The reason is as follows: our time limit was imposed for practical reasons to help with learning, but if the agent views the environment timing out as a form of failure that terminates its reward then it would have no reason to prefer the behaviour "stay perfectly level" to the behaviour "stay level for the first 499 timesteps then immediately fall over"! We want to encourage the agent to perform well all the time, not just perform well until the environment times out. See [this page](https://farama.org/Gymnasium-Terminated-Truncated-Step-API#theory) for more discussion.

We can illustrate this in a bit more depth. Consider the following diagram,
which illustrates a set of rollouts from 5 parallel copies of an environment:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/refs/heads/main/img/ch2-parallel_env_trajectories.png" width="700">

Here, we have 5 parallel environments, each run for 8 timesteps. Different colours
denote different episodes. Solid colour denotes the start of each episode, and a dotted
border indicates termination occurred during the transition from the previous observation to this one.
We can see immediately the (somewhat confusing) convention that when a trajectory terminates,
the observation received is the **initial observation of the next episode**. We can also see
the complexity that this poses in the variety of episodes

* **Env 1**: A single episode that runs for the length of the rollout over 7 timesteps, and terminates on timestep 8.
* **Env 2**: Two episodes, the first of length 3, which terminates (and on the same timestep we observe the starting state of the second), and the second terminates after 4 timesteps.
* **Env 3**: The same as Env 2, but the boundary between episodes doesn't match that of Env 2
* **Env 4**: Three episodes, but the third gets **truncated** rather than **terminated** (so the last observation is not drawn with a dotted border)
* **Env 5**: No episode terminates in time, so the only existing episode gets **truncated**.

You can see how this might be a problem to handle, especially if it's an environment where you only
receive reward at the end of the episode (and zero reward elsewhere).
Under these circumstances, Env 5 would give zero feedback at all for learning.

This is why we need the extra code that reads from `infos["final_observation"]`: given the last
observation $o_T$ for an episode, "$o_{T+1}$" as given from the vectorized environment will be the
starting state of the first episode, but what we actually want to learn from is
the true $o_{T+1}$ (the observation if the environment dynamics act upon $o_t$, ignoring termination).

### `ReplayBuffer` and `ReplayBufferSamples`

We've given you 2 classes below. The first, `ReplayBuffer`, holds data from past experiences and also contains methods for sampling that data (the samples are instances of `ReplayBufferSamples`).

You should read these implementations carefully, making sure you understand how they work. A few things to note:

- The `add` method adds multiple experiences at once: the tensors like `obs` have shape `(num_envs, *obs_shape)`. This is because our `CartPole` is a *vectorised* environment: a single call to `envs.step` advances all `num_envs` environments at once and returns batched tensors, one row per environment. We'll see how this works in practice later.
- The `add` method will add these experiences to the end of the buffer, slicing the buffer if it's too long. Note that the slicing is done so that we remove the oldest experiences when the buffer is full.
- The `sample` method will return a `ReplayBufferSamples` object containing the experiences sampled from the buffer. These are sampled with replacement. Everything in the buffer already lives on the GPU as tensors (the environment produces tensors, so there's no numpy anywhere in the loop), which means sampling is just indexing - no data ever has to be copied between the CPU and GPU.

In [20]:
@dataclass
class ReplayBufferSamples:
    """
    Samples from the replay buffer, as PyTorch tensors ready to feed into the Q-network.

    Data is equivalent to (s_t, a_t, r_{t+1}, d_{t+1}, s_{t+1}). Note - here, d_{t+1} is actually **terminated** rather
    than **done** (i.e. it records the times when we went out of bounds, not when the environment timed out).
    """

    obs: Float[Tensor, " sample_size *obs_shape"]
    actions: Int[Tensor, " sample_size"]
    rewards: Float[Tensor, " sample_size"]
    terminated: Bool[Tensor, " sample_size"]
    next_obs: Float[Tensor, " sample_size *obs_shape"]


class ReplayBuffer:
    """
    Contains buffer; has a method to sample from it to return a ReplayBufferSamples object.
    """

    rng: t.Generator
    obs: Float[Tensor, " buffer_size *obs_shape"]
    actions: Int[Tensor, " buffer_size"]
    rewards: Float[Tensor, " buffer_size"]
    terminated: Bool[Tensor, " buffer_size"]
    next_obs: Float[Tensor, " buffer_size *obs_shape"]

    def __init__(
        self,
        num_envs: int,
        obs_shape: tuple[int],
        action_shape: tuple[int],
        buffer_size: int,
        seed: int,
        device: t.device = device,
    ):
        self.num_envs = num_envs
        self.obs_shape = obs_shape
        self.action_shape = action_shape
        self.buffer_size = buffer_size
        self.device = t.device(device)
        self.rng = t.Generator(device=self.device).manual_seed(seed)

        self.obs = t.empty((0, *self.obs_shape), dtype=t.float32, device=self.device)
        self.actions = t.empty((0, *self.action_shape), dtype=t.int64, device=self.device)
        self.rewards = t.empty(0, dtype=t.float32, device=self.device)
        self.terminated = t.empty(0, dtype=t.bool, device=self.device)
        self.next_obs = t.empty((0, *self.obs_shape), dtype=t.float32, device=self.device)

    def add(
        self,
        obs: Float[Tensor, " num_envs *obs_shape"],
        actions: Int[Tensor, " num_envs"],
        rewards: Float[Tensor, " num_envs"],
        terminated: Bool[Tensor, " num_envs"],
        next_obs: Float[Tensor, " num_envs *obs_shape"],
    ) -> None:
        """
        Add a batch of transitions to the replay buffer.
        """
        # Check shapes & datatypes as `t.cat` silently *promotes* dtypes
        for name, data, expected_shape, expected_dtype in zip(
            ["obs", "actions", "rewards", "terminated", "next_obs"],
            [obs, actions, rewards, terminated, next_obs],
            [self.obs_shape, self.action_shape, (), (), self.obs_shape],
            [t.float32, t.int64, t.float32, t.bool, t.float32],
        ):
            assert isinstance(data, Tensor), f"`{name}` should be a tensor, got {type(data)}"
            assert data.shape == (self.num_envs, *expected_shape), f"`{name}` has shape {tuple(data.shape)}, expected {(self.num_envs, *expected_shape)}"
            assert data.dtype == expected_dtype, f"`{name}` has dtype {data.dtype}, expected {expected_dtype}"

        # Add data to buffer, slicing off the old elements
        self.obs = t.cat((self.obs, obs))[-self.buffer_size :]
        self.actions = t.cat((self.actions, actions))[-self.buffer_size :]
        self.rewards = t.cat((self.rewards, rewards))[-self.buffer_size :]
        self.terminated = t.cat((self.terminated, terminated))[-self.buffer_size :]
        self.next_obs = t.cat((self.next_obs, next_obs))[-self.buffer_size :]

    def sample(self, sample_size: int) -> ReplayBufferSamples:
        """
        Sample a batch of transitions from the buffer, with replacement.
        """
        indices = t.randint(0, self.obs.shape[0], (sample_size,), generator=self.rng, device=self.device)

        return ReplayBufferSamples(
            obs=self.obs[indices],                  # s_t
            actions=self.actions[indices],          # a_t
            rewards=self.rewards[indices],          # r_{t+1}
            terminated=self.terminated[indices],    # d_{t+1}: terminated (not truncated!)
            next_obs=self.next_obs[indices],        # s_{t+1}
        )

Next, you can run the following code to visualize your cart's position and angle, and see how these look in both the buffer and the buffer's random samples. Do the samples look correctly shuffled? Also, based on the [CartPole source code](https://github.com/Farama-Foundation/Gymnasium/blob/main/gymnasium/envs/classic_control/cartpole.py), do the angles & positions at which the cart terminates make sense? (Note, the min/max values in the table are different to the termination ranges, the latter can be found *below* the table in the docstring.)

Note that the code below uses our batched `CartPole` with `num_envs=1`, so every tensor it returns has a leading dummy batch dimension of size 1 (one row per environment).

<!-- Lastly, note how there's actually a slight epsiode mismatch between $s_t$ and $s_{t+1}$ in the plots below. The vertical lines show the values of $t$ where $d_{t+1} = 1$, so we can see that at these points $s_t$ and $s_{t+1}$ both refer to the terminated episode (they're the pre-final and final observations respectively), but immediately after these points $s_t$ refers to the terminal observation while $s_{t+1}$ refers to the first observation of the new episode. This isn't really a problem though, because we don't care what action our agent learns to take when $s_t$ is a terminal state (i.e. out of bounds). If we were being careful then we'd want to filter these terms out of our buffer, but for our purposes this week it doesn't really matter if we keep them in. -->

Lastly, note how when we terminate environments we do something slightly different. If `envs.step` results in some environments terminating, it'll actually return `next_obs` as the reset observation for the next episode. In this case, we want to use this as our starting observation for the next step, but we need to make sure we record the correct terminal observation in our buffer - we do this by extracting it from the `infos` dict, which is where it gets stored. You can see this in the plots below: the vertical lines are the values $t$ where $d_{t+1}=1$ i.e. $s_{t+1}$ is terminal, and we can see that $s_t$ is the final observation of the terminated episode while $s_{t+1}$ is the first observation of the new episode immediately after.

In [21]:
buffer = ReplayBuffer(num_envs=1, obs_shape=(4,), action_shape=(), buffer_size=256, seed=0, device=device)
envs = CartPole(num_envs=1, seed=0, device=device)
obs, infos = envs.reset()

for i in range(256):
    # Choose random actions (one per environment), and take a step in the environment
    actions = t.randint(0, envs.single_action_space.n, (envs.num_envs,), device=device)
    next_obs, rewards, terminated, truncated, infos = envs.step(actions)

    # Get `true_next_obs` by finding all environments where we terminated & replacing `next_obs`
    # with the actual terminal states (the env has already auto-reset those environments, and stored
    # the observation they ended on in `infos["final_observation"]`)
    done = terminated | truncated
    true_next_obs = next_obs.clone()
    true_next_obs[done] = infos["final_observation"][done]

    # Add experience to buffer
    buffer.add(obs, actions, rewards, terminated, true_next_obs)
    obs = next_obs

sample = buffer.sample(256)

plot_cartpole_obs_and_dones(
    buffer.obs.cpu(),
    buffer.terminated.cpu(),
    title="Current obs s<sub>t</sub><br>so when d<sub>t+1</sub> = 1, these are the states just before termination",
)

plot_cartpole_obs_and_dones(
    buffer.next_obs.cpu(),
    buffer.terminated.cpu(),
    title="Next obs s<sub>t+1</sub><br>so when d<sub>t+1</sub> = 1, these are the terminated states",
)

plot_cartpole_obs_and_dones(
    sample.obs.cpu(),
    sample.terminated.cpu(),
    title="Current obs s<sub>t</sub> (sampled)<br>this is what gets fed into our model for training",
)

# 2️⃣ Exploration & Probes

## Exploration

DQN makes no attempt to explore intelligently. The exploration strategy is the same as
for Q-Learning: agents take a random action with probability epsilon, but now we gradually
decrease epsilon. The Q-network is also randomly initialized (rather than initialized with zeros),
so its predictions of what is the best action to take are also pretty random to start.

Some games like [Montezuma's Revenge](https://paperswithcode.com/task/montezumas-revenge) have sparse rewards that require more advanced exploration methods to obtain. The player is required to collect specific keys to unlock specific doors, but unlike humans, DQN has no prior knowledge about what a key or a door is, and it turns out that bumbling around randomly has too low of a probability of correctly matching a key to its door. Even if the agent does manage to do this, the long separation between finding the key and going to the door makes it hard to learn that picking the key up was important.

As a result, DQN scored an embarrassing 0% of average human performance on this game.

### Reward Shaping

One solution to sparse rewards is to use human knowledge to define auxiliary reward functions that are more dense and made the problem easier (in exchange for leaking in side knowledge and making
the algorithm more specific to the problem at hand). What could possibly go wrong?

The canonical example is for a game called [CoastRunners](https://openai.com/blog/faulty-reward-functions/), where the goal was given to maximize the
score (hoping that the agent would learn to race around the map). Instead, it found it could
gain more score by driving in a loop picking up power-ups just as they respawn, crashing and
setting the boat alight in the process.

### Reward Hacking

For Montezuma's Revenge, the reward was shaped by giving a small reward for
picking up the key.
One time this was tried, the reward was given slightly too early and the agent learned it could go close to the key without quite picking it up, obtain the auxiliary reward, and then back up and repeat.

[![Montezuma Reward Hacking](https://img.youtube.com/vi/_sFp1ffKIc8/0.jpg)](https://www.youtube.com/watch?v=_sFp1ffKIc8 "Montezuma Reward Hacking")

A collected list of examples of Reward Hacking can be found [here](https://docs.google.com/spreadsheets/d/e/2PACX-1vRPiprOaC3HsCf5Tuum8bRfzYUiKLRqJmbOoC-32JorNdfyTiRRsR7Ea5eWtvsWzuxo8bjOxCG84dAg/pubhtml).


### Advanced Exploration

It would be better if the agent didn't require these auxiliary rewards to be hardcoded by humans,
but instead only relied on other signals from the environment that a state might be worth exploring. One idea is that a state which is "surprising" or "novel" (according to the agent's current belief
of how the environment works) in some sense might be valuable. Designing an agent to be
innately curious presents a potential solution to exploration, as the agent will focus exploration
in areas it is unfamiliar with. In 2018, OpenAI released [Random Network Distillation](https://openai.com/blog/reinforcement-learning-with-prediction-based-rewards/) which made progress in formalizing this notion, by measuring the agent's ability to predict the output of a neural network
on visited states. States that are hard to predict are poorly explored, and thus highly rewarded.
In 2019, an excellent paper [First return, then explore](https://arxiv.org/pdf/2004.12919v6.pdf) found an even better approach. Such reward shaping can also be gamed, leading to the
noisy TV problem, where agents that seek novelty become entranced by a source of randomness in the
environment (like a analog TV out of tune displaying white noise), and ignore everything else
in the environment.

### Exercise - implement linear scheduler

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Importance: 🔵🔵⚪⚪⚪
>
> You should spend up to 5-10 minutes on this exercise.
> ```

For now, implement the basic linearly decreasing exploration schedule.

In [22]:
def linear_schedule(
    current_step: int,
    start_e: float,
    end_e: float,
    exploration_fraction: float,
    total_timesteps: int,
) -> float:
    """
    Return the appropriate epsilon for the current step.

    Epsilon should be start_e at step 0 and decrease linearly to end_e at step (exploration_fraction
    * total_timesteps). In other words, we are in "explore mode" with start_e >= epsilon >= end_e
    for the first `exploration_fraction` fraction of total timesteps, and then stay at end_e for the
    rest of training.
    """
    return max((end_e - start_e)/ (exploration_fraction *total_timesteps)*current_step + start_e, end_e)



epsilons = [
    linear_schedule(step, start_e=1.0, end_e=0.05, exploration_fraction=0.5, total_timesteps=500)
    for step in range(500)
]
line(
    epsilons,
    labels={"x": "steps", "y": "epsilon"},
    title="Probability of random action",
    height=400,
    width=600,
)

tests.test_linear_schedule(linear_schedule)

All tests in `test_linear_schedule` passed!


<details><summary>Solution</summary>

```python
def linear_schedule(
    current_step: int,
    start_e: float,
    end_e: float,
    exploration_fraction: float,
    total_timesteps: int,
) -> float:
    """
    Return the appropriate epsilon for the current step.

    Epsilon should be start_e at step 0 and decrease linearly to end_e at step (exploration_fraction
    * total_timesteps). In other words, we are in "explore mode" with start_e >= epsilon >= end_e
    for the first `exploration_fraction` fraction of total timesteps, and then stay at end_e for the
    rest of training.
    """
    return start_e + (end_e - start_e) * min(current_step / (exploration_fraction * total_timesteps), 1)
```
</details>

### Epsilon Greedy Policy

In DQN, the policy is implicitly defined by the Q-network: we take the action with the maximum predicted reward. This gives a bias towards optimism. By estimating the maximum of a set of values $v_1, \ldots, v_n$ using the maximum of some noisy estimates $\hat{v}_1, \ldots, \hat{v}_n$ with $\hat{v}_i \approx v_i$, we get unlucky and get very large positive noise on some samples, which the maximum then chooses. Hence, the agent will choose actions that the Q-network is overly optimistic about.

See Sutton and Barto, Section 6.7 if you'd like a more detailed explanation, or the original [Double Q-Learning](https://proceedings.neurips.cc/paper/2010/file/091d584fced301b442654dd8c23b3fc9-Paper.pdf) paper which notes this maximisation bias, and introduces a method to correct for it using two separate Q-value estimators, each used to update the other.

### Exercise - implement the epsilon greedy policy

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
>
> You should spend up to 10-20 minutes on this exercise.
> ```

Each environment decides *independently* whether to explore: environment $i$ takes a uniformly random action with probability $\varepsilon$, and the greedy action $\arg\max_a Q(o_t, a)$ otherwise.
Since some environments will almost always want the greedy action, just always do the forward pass, then use `t.where` to pick per environment.

<details>
<summary> Alternate definition of exploration </summary>

We could have done this another way: With probability $\epsilon$,
the policy for **all** environments explores, else the policy takes
the greedy action for **all** environments. This has the slight advantage that
it allows you to skip a forward pass during the exploration step, but
it also makes separate **episodes** correlated: we get bursts of either
$0$ explore, or $n$ explore steps, rather than roughly
$n\epsilon$ explore steps. The expectation is the same, but the variance is
much higher.

The natural approach is to consider the biased coin-flips as independent.
</details>


Other tips:

- Work with the torch random number generator `rng` that we've provided, rather than the global RNG. `t.rand(size, generator=rng, device=device)` gives uniform random numbers in $[0,1)$, and `t.randint(0, n, size, generator=rng, device=device)` gives random integers in the range $0, 1, \ldots, n-1$.
- Use `envs.single_action_space.n` to get the number of actions.

In [23]:
def epsilon_greedy_policy(
    envs: CartPole,
    q_network: QNetwork,
    rng: t.Generator,
    obs: Float[Tensor, " num_envs *obs_shape"],
    epsilon: float,
) -> Int[Tensor, " num_envs"]:
    """
    For each environment independently: with probability epsilon take a random action, otherwise take the greedy
    action according to the q_network.

    Inputs:
        envs:       The batched environment we're acting in
        q_network:  The QNetwork used to approximate the Q-value function
        rng:        A torch random number generator
        obs:        The current observation for each environment, shape (num_envs, *obs_shape)
        epsilon:    The probability of taking a random action

    Returns:
        actions:    The sampled action for each environment, shape (num_envs,).
    """
    rands = t.rand(envs.num_envs, generator=rng, device=device)
    random_actions = t.randint(0, int(envs.single_action_space.n), (envs.num_envs,)).to(device)
    Q_actions = t.argmax(q_network.forward(obs), dim=1)
    return t.where(rands < epsilon, random_actions, Q_actions)


tests.test_epsilon_greedy_policy(epsilon_greedy_policy)

All tests in `test_epsilon_greedy_policy` passed!


<details>
<summary>Help - I'm confused about the action shape here.</summary>

In our case, the action shape is `envs.single_action_space.shape = ()` (i.e. trivial, because our action is just a single integer not a vector or tensor) and the number of possible actions is `envs.single_action_space.n = 2`. This means that `*action_shape` is empty, and your return type should just be a tensor of ints of shape `(num_envs,)` with each element being uniformly sampled from `[0, 1]`.

More generally, we could consider the action space continuous, where the action is a tensor of shape `(*action_space,)`
which could be multi-dimensional, and then the return type would be `(num_envs, *action_space)`. We will
see continuous actions spaces on PPO day.
</details>


<details><summary>Solution</summary>

```python
def epsilon_greedy_policy(
    envs: CartPole,
    q_network: QNetwork,
    rng: t.Generator,
    obs: Float[Tensor, " num_envs *obs_shape"],
    epsilon: float,
) -> Int[Tensor, " num_envs"]:
    """
    For each environment independently: with probability epsilon take a random action, otherwise take the greedy
    action according to the q_network.

    Inputs:
        envs:       The batched environment we're acting in
        q_network:  The QNetwork used to approximate the Q-value function
        rng:        A torch random number generator
        obs:        The current observation for each environment, shape (num_envs, *obs_shape)
        epsilon:    The probability of taking a random action

    Returns:
        actions:    The sampled action for each environment, shape (num_envs,).
    """
    num_envs = obs.shape[0]
    num_actions = envs.single_action_space.n
    with t.no_grad():
        greedy_actions = q_network(obs).argmax(dim = -1)
    is_explore = t.rand(num_envs, generator=rng, device=obs.device) < epsilon
    random_actions = t.randint(0, num_actions, (num_envs,), generator=rng, device=obs.device)
    return t.where(is_explore, random_actions, greedy_actions)
```
</details>

## Probe Environments

Extremely simple probe environments are a great way to debug your algorithm. Ours live in `chapter2_rl/exercises/gpu_probe.py`, but we've copied one in for your convenience.


```python
# Don't copy in this code, just for a demonstration
class Probe1(gym.Env):
    """
    Vectorized Probe1: One action, observation of [0.0], one timestep long, +1 reward.

    We expect the agent to rapidly learn that the value of the constant [0.0] observation is +1.0.
    Note we're using a continuous observation space for consistency with CartPole.
    """

    action_space: Discrete
    observation_space: Box

    def __init__(self, num_envs: int = 3, seed: int = 0, device: str | t.device = device):
        super().__init__()
        self.num_envs = num_envs
        self.device = t.device(device)

        self.observation_space = Box(np.array([0.0], dtype=np.float32), np.array([0.0], dtype=np.float32))
        self.action_space = Discrete(1)
        self.single_observation_space = self.observation_space  # vector-env style aliases
        self.single_action_space = self.action_space

        self.state = t.zeros([self.num_envs, 1], dtype=t.float32, device=self.device)

    def step(self, action: t.Tensor) -> tuple[t.Tensor, t.Tensor, t.Tensor, t.Tensor, dict]:
        reward = t.ones([self.num_envs], dtype=t.float32, device=self.device)
        terminated = t.ones([self.num_envs], dtype=t.bool, device=self.device)
        truncated = t.zeros([self.num_envs], dtype=t.bool, device=self.device)

        # Every env terminates each step; the auto-reset state is identical ([0.0]).
        infos = {"final_observation": self.state.clone()}
        return self.state.clone(), reward, terminated, truncated, infos

    def reset(self, *, seed: int | None = None, options=None) -> tuple[t.Tensor, dict]:
        self.state = t.zeros([self.num_envs, 1], dtype=t.float32, device=self.device)
        return self.state.clone(), {}

```

Let's try and break down how this environment works. We see that the function `step` always returns the same thing. The observation and reward are always the same, and `terminated` is always true (i.e. the episode always terminates after one action). We expect the agent to rapidly learn that the value of the constant observation `[0.0]` is `+1`. This is in some sense the simplest possible probe.

### A note on observation spaces

The space we're using here is `gym.spaces.Box`. This means we're dealing with real-valued quantities, i.e. continuous not discrete. The first two arguments of `Box` are `low` and `high`, and these define a box in $\mathbb{R}^n$. For instance, if these arrays are `(0, 0)` and `(1, 1)` respectively, this defines the box $0 \leq x, y \leq 1$ in 2D space.

For this probe, the observation space is a degenerate box, zero-dimensional point at the origin.

In [24]:
env = Probe1(num_envs=4)
assert env.single_observation_space.shape == (1,)
assert env.single_action_space.shape == ()
obs, _ = env.reset()
assert obs.shape == (4, 1)

### Exercise - read & understand other probe environments

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵🔵⚪
>
> You should spend up to 10-20 minutes here.
> ```

For each of the probes in `gpu_probe.py`, read their implementation code, and understand how they correspond to their docstrings (and to the [descriptions](https://andyljones.com/posts/rl-debugging.html#:~:text=Use%20probe%20environments) given in Andy Jones' post).

It's very important to understand how these probes work, and why they're useful tools for debugging. When you're working on your own RL projects, you might have to write your own probes to suit your particular use cases.

A brief summary of these, along with recommendations of where to go to debug if one of them fails (note that these won't be true 100% of the time, but should hopefully give you some useful direction):

<details>
<summary>Summary of probes</summary>

1. **Tests basic learning ability**. If this fails, it means the agent has failed to learn to associate a constant observation with a constant reward. You should check your loss functions and optimizers in this case.
2. **Tests the agent's ability to differentiate between 2 different observations (and learn their respective values)**. If this fails, it means the agent has issues with handling multiple possible observations.
3. **Tests the agent's ability to handle time & reward delay**. If this fails, it means the agent has problems with multi-step scenarios of discounting future rewards. You should look at how your agent step function works.
4. **Tests the agent's ability to learn from actions leading to different rewards**. If this fails, it means the agent has failed to change its policy for different rewards, and you should look closer at how your agent is updating its policy based on the rewards it receives & the loss function.
5. **Tests the agent's ability to map observations to actions**. If this fails, you should look at the code which handles multiple timesteps, as well as the code that handles the agent's map from observations to actions.

</details>

# 3️⃣ Main DQN Algorithm

## Main DQN Algorithm

We now combine all the elements we have designed thus far into the final DQN algorithm. Here, we assume the environment returns three parameters $(s_{new}, r, d)$, a new state $s_{new}$, a reward $r$ and a boolean $d$ indicating whether interaction has terminated yet.

Our Q-value function $Q(s,a)$ is now a network $Q(s,a ; \theta)$ parameterised by weights $\theta$. The key idea, as in Q-learning, is to ensure the Q-value function satisfies the optimal Bellman equation
$$
Q(s,a ; \theta)
= \mathbb{E}_{s',r \sim p(\cdot \mid s,a)} \left[r + \gamma \max_{a'} Q(s', a' ;\theta) \right]
$$
which means the expected TD error will be zero (where expectation here is taken over randomly sampled trajectories):
$$
\mathbb{E} \left[ r_{t+1} + \gamma \max_a Q(s_{t+1}, a) - Q(s_t, a_t) \right] = 0
$$
Note that we also have to deal with the case where the episode terminates: i.e. $s_{t+1}$ is a terminal state, and $d_{t+1} = 1$. Since the Q-value of a terminal state is always zero, we can just rewrite the expected TD error expression as:
$$
\mathbb{E} \left[ r_{t+1} + (1 - d_{t+1}) \, \gamma \max_a Q(s_{t+1}, a) - Q(s_t, a_t) \right] = 0
$$
since this makes sure the term is zero whenever $d_{t+1} = 1$. We can now see again why we used `terminated` not `terminated | truncated` here - we don't want the agent to learn that its value is always zero just before the episode ends and so there's no point in continuing to perform well!

Since we have an expression which should be zero in expectation for our true Q-value function, and we want the model to learn from a variety of experiences at once, we can sample batches of experiences $B = \{s_{t_i}, a_{t_i}, r_{t_i+1}, d_{t_i+1}, s_{t_i+1}\}_{i=1}^{|B|}$ from the replay buffer, and train against the loss function which equals the **squared temporal difference error**:
$$
L(\theta) = \frac{1}{|B|} \sum_{i=1}^{|B|} \left( r_{t_i+1} + (1 - d_{t_i+1}) \gamma \max_a Q(s_{t_i+1}, a ; \theta_\text{target}) - Q(s_{t_i}, a_{t_i} ; \theta) \right)^2
$$
Here, $\theta_\text{target}$ is a previous copy of the parameters $\theta$, so we're updating our $s_t$ estimates to catch up with our $s_{t+1}$ estimates (just like in standard Q-learning from earlier!). Every so often, we then update the target $\theta_\text{target} \leftarrow \theta$ as the agent improves its Q-values from experience.

Below is the full DQN algorithm from a paper, for reference. The notation isn't identical to ours (e.g. they use an if/else statement to handle the terminal state case), but the basic algorithm is the same.

<img src="https://raw.githubusercontent.com/chloeli-15/ARENA_img/refs/heads/main/img/ch2-dqn-algo.png" width="700">

### DQN Dataclass

Below is a dataclass for training your DQN - read through it to see what you'll be working with (the arguments are grouped by what they control).

The exact breakdown of training is as follows:

* The agent takes `total_timesteps` steps in the environment during the training loop. These are *environment* steps, summed over all of our `num_envs` parallel environments: each call to `agent.play_step` advances every environment once, so we advance the total timesteps counter by `num_envs` on each step.
* The first `buffer_size` of these steps are used to fill the replay buffer (we don't update gradients until the buffer is full).
* After this point, we perform an optimizer step every `steps_per_train` calls to `agent.play_step` (i.e. every `steps_per_train * num_envs` environment steps). We also copy the weights from our Q-network to our target network every `trains_per_target_update` steps of our Q-network.

This is shown in the diagram below (the actual numbers aren't representative of the values in our dataclass, they're just to make sure the diagram is understandable - obviously the scale is very different in our actual training).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/dqn-breakdown.png" width="1200">

For example, in the code below we change `total_timesteps`, and this also changes the total number of training steps (which is computed in the `__post_init__` method of our dataclass, as a function of `total_timesteps`).

In [25]:
@dataclass
class DQNArgs:
    # Basic / global
    seed: int = 1
    env_id: str = "CartPole-gpu"
    num_envs: int = 16 # number of parallel environments to run
    device: str = device

    # Wandb / logging
    use_wandb: bool = False
    wandb_project_name: str = "DQNCartPole"
    wandb_entity: str | None = None
    video_log_freq: int | None = None # number of training steps between wandb video upload
    steps_per_live_video: int | None = 5_000 # number of training steps between live video display
    overwrite_video: bool = True # older live videos are overwritten by newer ones if True. Otherwise, they are plotted one after the other.

    # Duration of different phases / buffer memory settings
    total_timesteps: int = 300_000
    steps_per_update: int = 1
    trains_per_target_update: int = 50
    buffer_size: int = 10_000

    # Optimization hparams
    batch_size: int = 512
    learning_rate: float = 5e-3

    # RL-specific
    gamma: float = 0.99
    exploration_fraction: float = 0.1
    start_e: float = 1.0
    end_e: float = 0.02

    def __post_init__(self):
        assert self.total_timesteps - self.buffer_size >= self.steps_per_update * self.num_envs
        # Every call to `agent.play_step` advances all `num_envs` environments by one step
        self.total_training_steps = (self.total_timesteps - self.buffer_size) // (self.steps_per_update * self.num_envs)
        self.device = t.device(self.device)

### Exercise - fill in the agent class

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵🔵⚪
>
> You should spend up to 25-45 minutes on this exercise.
> ```

You should now fill in the methods for the `DQNAgent` class below. This is a class which is designed to handle taking steps in the environment (with an epsilon greedy policy), and updating the buffer.

1. `play_step` should be somewhat similar to the demo code you saw earlier, which sampled a batch of experiences to add to the buffer. It should:
    - Get actions (using `self.get_actions` rather than randomly sampling like we did in the demo code before)
    - Step our environment with these actions
    - Add the new experiences to the buffer: remembering that for environments which just finished an episode, `next_obs` is already the reset observation, and the true final observation is in `infos["final_observation"]`.
    - Set your new observation as `self.obs`, ready for the next step
2. `get_actions` should do the following:
    - Set `self.epsilon` according to the linear schedule function & the current global step counter
    - Sample actions according to the epsilon-greedy policy (i.e. using your `epsilon_greedy_policy` function), and return them

A small note on code practices here - the implementation below was designed to follow **separation of concerns** (SoC), a design principle used in software engineering. The `DQNAgent` class is only responsible for interacting with the environment; it doesn't do anything like create the Q-network or buffer on initialization. This is further reflected in the fact that we don't pass in `args` to our DQN agent, but instead pass in all the relevant variables separately (if we were forced to pass in `args`, this would be a sign that the DQN agent class might be doing too much work!).

In [26]:
class DQNAgent:
    """Base Agent class handling the interaction with the environment."""

    def __init__(
        self,
        envs: CartPole,
        buffer: ReplayBuffer,
        q_network: QNetwork,
        start_e: float,
        end_e: float,
        exploration_fraction: float,
        total_timesteps: int,
        rng: t.Generator,
    ):
        self.envs = envs
        self.buffer = buffer
        self.q_network = q_network
        self.start_e = start_e
        self.end_e = end_e
        self.exploration_fraction = exploration_fraction
        self.total_timesteps = total_timesteps
        self.rng = rng

        self.step = 0  # Tracking number of steps taken (across all environments)
        self.obs, _ = self.envs.reset()  # Need a starting observation
        self.epsilon = start_e  # Starting value (will be updated in `get_actions`)

    def play_step(self) -> dict:
        """
        Carries out a single interaction step between agent & environment, and adds results to the
        replay buffer.

        Returns `infos` (a dict containing info we will log).
        """
        actions = self.get_actions(self.obs)
        state, reward, terminated, truncated, infos = self.envs.step(actions)
        done = terminated | truncated

        true_next_obs = state.clone()
        true_next_obs[done] = infos["final_observation"][done]
        self.buffer.add(self.obs, actions, reward, terminated, true_next_obs)

        self.obs = state

        self.step += self.envs.num_envs
        return infos

    def get_actions(self, obs: Float[Tensor, " num_envs *obs_shape"]) -> Int[Tensor, " num_envs"]:
        """
        Samples actions according to the epsilon-greedy policy using the linear schedule for epsilon.
        """
        self.epsilon = linear_schedule(self.step, self.start_e, self.end_e, self.exploration_fraction, self.total_timesteps)
        return epsilon_greedy_policy(self.envs, self.q_network, self.rng, obs, self.epsilon)


tests.test_agent(DQNAgent)

All tests in `test_agent` passed!


<details><summary>Solution</summary>

```python
class DQNAgent:
    """Base Agent class handling the interaction with the environment."""

    def __init__(
        self,
        envs: CartPole,
        buffer: ReplayBuffer,
        q_network: QNetwork,
        start_e: float,
        end_e: float,
        exploration_fraction: float,
        total_timesteps: int,
        rng: t.Generator,
    ):
        self.envs = envs
        self.buffer = buffer
        self.q_network = q_network
        self.start_e = start_e
        self.end_e = end_e
        self.exploration_fraction = exploration_fraction
        self.total_timesteps = total_timesteps
        self.rng = rng

        self.step = 0  # Tracking number of steps taken (across all environments)
        self.obs, _ = self.envs.reset()  # Need a starting observation
        self.epsilon = start_e  # Starting value (will be updated in `get_actions`)

    def play_step(self) -> dict:
        """
        Carries out a single interaction step between agent & environment, and adds results to the
        replay buffer.

        Returns `infos` (a dict containing info we will log).
        """
        actions = self.get_actions(self.obs)
        next_obs, rewards, terminated, truncated, infos = self.envs.step(actions)

        # Get `true_next_obs` by finding all environments where we terminated & replacing `next_obs`
        # with the actual terminal states (the env has already auto-reset those environments)
        done = terminated | truncated
        true_next_obs = next_obs.clone()
        true_next_obs[done] = infos["final_observation"][done]

        self.buffer.add(self.obs, actions, rewards, terminated, true_next_obs)
        self.obs = next_obs

        self.step += self.envs.num_envs
        return infos

    def get_actions(self, obs: Float[Tensor, " num_envs *obs_shape"]) -> Int[Tensor, " num_envs"]:
        """
        Samples actions according to the epsilon-greedy policy using the linear schedule for epsilon.
        """
        self.epsilon = linear_schedule(
            self.step, self.start_e, self.end_e, self.exploration_fraction, self.total_timesteps
        )
        actions = epsilon_greedy_policy(self.envs, self.q_network, self.rng, obs, self.epsilon)
        assert actions.shape == (self.envs.num_envs,)
        return actions
```
</details>

Before we move on to the big exercise of today (completing the `DQNTrainer` class), we'll briefly discuss logging to Weights and Biases in RL, plus some general advice on what kinds of variables you should be logging.

### Logging to `wandb` in RL

In previous exercises in this chapter, we've just trained the agent, and then plotted the reward per episode after training. For small toy examples that train in a few seconds this is fine, but for longer runs we'd like to watch the run live and make sure the agent is doing something interesting (especially if we were planning to run the model overnight). Luckily, **Weights and Biases** has got us covered! When you run your experiments, you'll be able to view not only *live plots* of the loss and average reward per episode while the agent is training - you can also log and view animations, which visualise your agent's progress in real time! The code below will handle all logging.

Sadly, effective logging & debugging in RL isn't just about watching videos, since in the vast majority of cases where your algorithm has a bug, the agent will just fail to learn anything useful and the videos won't be informative. Debugging RL requires knowing what variables to log and how to interpret the results you're getting, which requires some understanding of the underlying theory! This is part of the reason why we've spent so much time discussing the theory behind DQN and other RL algorithms, rather than just giving you a black box to train.

As an example of how logged variables can be misleading and hard to interpret, consider our TD loss function in DQN. This loss function just reflects how close together the Q-network's estimates are to the experiences currently sampled from the replay buffer, which might not adequately represent what the world actually looks like. This means that once the agent starts to learn something and do better at the problem, it's expected for the loss to increase. For example, maybe the Q-network initially learned some state was bad, because an agent that reached them was just flapping around randomly and died shortly after. But now it's getting evidence that the same state is good, now that the agent that reached the state has a better idea what to do next. A higher loss is thus actually a good sign that something is happening (the agent hasn't stagnated), but it's not clear if it's learning anything useful without also checking how the total reward per episode has changed. Key point - **just looking at one variable can be misleading, we need to log multiple variables and derive a picture of what's happening from taking all of them into account!**

Some useful variables to log during DQN training are:

- TD loss, i.e. the actual loss you're backpropagating through. This should start off high and decrease pretty quickly, but may not be monotonic (i.e. temporary spikes in loss aren't necessarily a bad thing)
- SPS (steps per second), i.e. the total number of agent steps divided by the total time. This helps us debug when the environment steps are a bottleneck (won't be the case in a simple environment like this one, but might matter more when we move to more complex environments)
- Q-values, i.e. the predicted Q-values from the Q-network. Can you guess how these should behave?

<details>
<summary>Question - what do you think the Q values will do when the agent moves closer to solving the cartpole environment?</summary>

Initially they should be near zero, thanks to the randomly initialized model weights. As our episode length get closer to 500 (i.e. we can essentially solve the environment), they should tend to the limit of the total possible time-discounted reward available, which is the geometric sum $1 + \gamma + \gamma^2 + \cdots$ (since we get 1 reward for every second we stand up, and as previously discussed, the way we handle `dones` in the formula above doesn't assume a truncated environment causes future rewards to be terminated). The limit of this sum is $\frac{1 - \gamma^{500}}{1 - \gamma} \approx \frac{1}{1-\gamma}$, which for our default value $\gamma = 0.99$ is approximately 100. Intuitively, the agent isn't "planning" far enough into the future to care about the full 500
timesteps, only the next 100. But this is good enough to ensure sensible behaviour (agents that tend to optimise well over
the next 100 steps end up doing the same thing as agents with $\gamma \to 1$.)

Note, the Q values won't increase smoothly, they'll spike up immediately after we copy over the weights from our Q-network to our target network. This is because each time we copy over weights, our gradient changes and the Q-network rapidly "catches up" to this new target network, causing the Q values to change rapidly. However, our copying over of weights will be frequent enough that these jumps will be relatively small, and so the curve should still appear smooth.

</details>

### Exercise - write DQN training loop

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Importance: 🔵🔵🔵🔵🔵
>
> You should spend up to 30-60 minutes on this exercise.
> ```

Now we'll create a new class `DQNTrainer`, which will handle the full training loop. We've filled in the `__init__` for you, which defines all the things you need (the networks, optimizer, replay buffer, and the agent). We've also filled in `train` for you, which performs the main training loop: it optionally initializes Weights & Biases, fills the buffer using `prepopulate_replay_buffer`, then alternates between training steps (where we sample from the buffer) & adding to the buffer (adding `args.steps_per_update * num_envs` new experiences each time).

You should fill in the remaining 2 methods. First you should get the basic no-logging version working, then once you're running without error (even if maybe you're not learning anything useful) you should move onto logging as this will help you debug.

- `add_to_replay_buffer`
    - This calls `self.agent.play_step()` to take `n` steps in the environment, which adds the results to the replay buffer
    - It's used to fill the buffer before training starts, and before each training step to add new experiences to the buffer
- `training_step`
    - This performs an update step from a batch of experiences from the buffer, sampled using `self.buffer.sample` with batch size `self.args.batch_size`
    - An update step involves:
        - Getting the predicted Q-values $Q(s_{t_i}, a_{t_i} ; \theta)$ from the Q-network
        - Getting the max target Q-values $\max_a Q(s_{t_i+1}, a ; \theta_\text{target})$ from the target network (remember to use inference mode - we're not training the target network!)
        - Computing the TD loss $L(\theta)$ using the formula we gave earlier (we've also copied it below, for convenience)
        - Performing an update step with this loss
    - You should also copy weights from the Q-network to the target network every `args.trains_per_target_update` steps (i.e. whenever the `step` argument is a multiple of this). The `load_state_dict` method might be useful here

For convenience, here's the full TD loss formula again:
$$
L(\theta) = \frac{1}{|B|} \sum_{i=1}^{|B|} \left( r_{t_i+1} + (1 - d_{t_i+1}) \gamma \max_a Q(s_{t_i+1}, a ; \theta_\text{target}) - Q(s_{t_i}, a_{t_i} ; \theta) \right)^2
$$

When you get to logging, there are 2 types of data you can log:

- Data for terminated episodes, during buffer filling
    - Terminated episode data can be found in the `infos` dict returned by the `agent.play_step` method. If at least one environment finished an episode on that step, then `infos["final_info"][0]["episode"]` will be a dict containing the length `l` and reward `r` of the finished episode(s), averaged over the environments that finished on that step (this is produced for our GPU env by `track_episode_stats` in `gpu_env.py`, and mirrors the layout gymnasium `0.29`'s vector environments used with the `RecordEpisodeStatistics` wrapper)
        - We've given you a helper function `get_episode_data_from_infos` which gives you a dict of the episode length & reward, or `None` if no envs terminated. See `gpu_env.py` for its implementation, and gymnasium's [`RecordEpisodeStatistics` documentation](https://gymnasium.farama.org/api/wrappers/misc_wrappers/#gymnasium.wrappers.RecordEpisodeStatistics) for the wrapper this format is based on (note that gymnasium 1.x vector environments lay these stats out differently, as `infos["episode"]` with an `infos["_episode"]` mask).
    - `add_to_replay_buffer` should just *return* this dict for the last episode that finished: the training loop we've given you logs it to the progress bar and to wandb (along with the SPS, steps per second) - at most twice a second
- Data during training steps
    - Mean TD loss, Q values, and the epsilon hyperparameter are all useful to log, but log them every 10 or so training steps rather than every step. A `wandb.log` call costs about as much as a whole gradient step on this tiny network, so logging on every step would more than double the run time!

Don't be discouraged if your code takes a while to work - it's normal for debugging RL to take longer than you would expect. Add asserts or your own tests, implement an appropriate probe environment, try anything in the Andy Jones post that sounds promising, and try to notice confusion. Reinforcement Learning is often so tricky as even if the algorithm has bugs, the agent might still learn something useful regardless (albeit maybe not as well), or even if everything is correct, the agent might just fail to learn anything useful (like how DQN failed to do anything on Montezuma's Revenge.)

Since the environment is already known to be one DQN can solve, and we've already provided hyperparameters that work for this environment, hopefully that's isolated a lot of the problems one would usually have with solving real world problems with RL.

In [40]:
def get_episode_data_from_infos(infos: dict) -> dict[str, int | float] | None:
    """
    Helper function: returns a dict of episode data (length, reward and duration, averaged over the
    environments that finished an episode on this step), if at least one
    terminated.
    """
    for final_info in infos.get("final_info", []):
        if final_info is not None and "episode" in final_info:
            return {
                "episode_length": final_info["episode"]["l"].item(),
                "episode_reward": final_info["episode"]["r"].item(),
                "episode_duration": final_info["episode"]["t"].item(),
            }


class DQNTrainer:
    def __init__(self, args: DQNArgs):
        set_global_seeds(args.seed)
        self.args = args
        self.rng = t.Generator(device=args.device).manual_seed(args.seed)
        self.run_name = f"{args.env_id}__{args.wandb_project_name}__seed{args.seed}__{time.strftime('%Y%m%d-%H%M%S')}"

        self.envs = make_envs(args.env_id, args.num_envs, args.seed, args.device)

        # Define some basic variables from our environment (note, we assume a single discrete action space)
        num_envs = self.envs.num_envs
        action_shape = self.envs.single_action_space.shape
        num_actions = self.envs.single_action_space.n
        obs_shape = self.envs.single_observation_space.shape
        assert action_shape == ()

        # Create our replay buffer (on the same device as the environment)
        self.buffer = ReplayBuffer(num_envs, obs_shape, action_shape, args.buffer_size, args.seed, args.device)

        # Create our networks & optimizer (target network should be initialized with a copy of the Q-network's weights)
        self.q_network = QNetwork(obs_shape, num_actions).to(args.device)
        self.target_network = QNetwork(obs_shape, num_actions).to(args.device)
        self.target_network.load_state_dict(self.q_network.state_dict())
        self.optimizer = t.optim.AdamW(self.q_network.parameters(), lr=args.learning_rate)

        # Create our agent
        self.agent = DQNAgent(
            self.envs,
            self.buffer,
            self.q_network,
            args.start_e,
            args.end_e,
            args.exploration_fraction,
            args.total_timesteps,
            self.rng,
        )

    def add_to_replay_buffer(self, n: int, verbose: bool = False):
        """
        Takes n steps with the agent, adding to the replay buffer. Should return a dict of data from
        the last terminated episode, if any (the training loop takes care of logging it).

        Optional argument `verbose`: if True, we can use a progress bar (useful to check how long
        the initial buffer filling is taking).
        """
        iter = range(n)
        if verbose:
            iter = tqdm(iter)
        ret = None
        for _ in iter:
            ret = self.agent.play_step()
        return ret

    def prepopulate_replay_buffer(self):
        """
        Called to fill the replay buffer before training starts.
        """
        n_steps_to_fill_buffer = self.args.buffer_size // self.args.num_envs
        self.add_to_replay_buffer(n_steps_to_fill_buffer, verbose=True)

    def training_step(self, step: int) -> None:
        """
        Samples once from the replay buffer, and takes a single training step.

        Args:
            step (int): The number of training steps taken (used for logging, and for deciding when
            to update the target network)
        """

        s = self.buffer.sample(self.args.batch_size)
        obs, action, rewards, terminated, next_obs = s.obs, s.actions, s.rewards, s.terminated, s.next_obs
        Q_pred = self.q_network(obs)[t.arange(len(action)), action]
        max_target = self.target_network(next_obs).max(dim=1).values
        loss_term = (rewards + (1-terminated.int())*self.args.gamma*max_target -Q_pred)**2
        loss = 1/self.args.batch_size * loss_term.sum()
        loss.backward()
        self.optimizer.step()
        self.optimizer.zero_grad()

        if step % self.args.trains_per_target_update == 0:
            self.target_network.load_state_dict(self.q_network.state_dict())



    def train(self) -> None:
        if self.args.use_wandb:
            wandb.init(
                project=self.args.wandb_project_name,
                entity=self.args.wandb_entity,
                name=self.run_name,
            )

        self.prepopulate_replay_buffer()

        pbar = tqdm(range(self.args.total_training_steps))
        last_logged_time, last_logged_step = time.time(), self.agent.step  # so we don't log too often

        greedy = lambda obs: self.q_network(obs).argmax(-1)
        fresh_envs = lambda seed: CartPole(num_envs=16, seed=seed, device=self.args.device)
        can_render = self.args.env_id == "CartPole-gpu"
        live_videos = can_render and self.args.steps_per_live_video is not None
        wandb_videos = can_render and self.args.use_wandb and self.args.video_log_freq is not None
        video_slot = LiveVideo(overwrite=self.args.overwrite_video)  # one display slot per run (see rl_utils.LiveVideo)

        for step in pbar:
            data = self.add_to_replay_buffer(self.args.steps_per_update)
            if data is not None:
                if time.time() - last_logged_time > 0.5:
                    now = time.time()
                    if self.args.use_wandb:
                        sps = (self.agent.step - last_logged_step) / (now - last_logged_time)
                        wandb.log({**data, "SPS": sps}, step=self.agent.step)
                    last_logged_time, last_logged_step = now, self.agent.step
                    pbar.set_postfix(**data)

            self.training_step(step)

            live = live_videos and step % self.args.steps_per_live_video == 0
            to_wandb = wandb_videos and step % self.args.video_log_freq == 0
            if live or to_wandb:
                log_greedy_rollout_video(fresh_envs(step), greedy, self.envs.draw, step=self.agent.step, live=live, slot=video_slot)


        self.envs.close()
        if self.args.use_wandb:
            wandb.finish()

<details>
<summary>Solution (simple, no logging)</summary>

```python
def add_to_replay_buffer(self, n: int, verbose: bool = False):
    """
    Takes n steps with the agent, adding to the replay buffer (and logging any results). Should return a dict of
    data from the last terminated episode, if any.

    Optional argument `verbose`: if True, we can use a progress bar (useful to check how long the initial buffer
    filling is taking).
    """
    data = None

    for step in tqdm(range(n), disable=not verbose, desc="Adding to replay buffer"):
        infos = self.agent.play_step()
        new_data = get_episode_data_from_infos(infos)
        data = new_data if new_data is not None else data  # keep the *last* terminated episode

    return data

def prepopulate_replay_buffer(self):
    """
    Called to fill the replay buffer before training starts.
    """
    n_steps_to_fill_buffer = self.args.buffer_size // self.args.num_envs
    self.add_to_replay_buffer(n_steps_to_fill_buffer, verbose=True)

def training_step(self, step: int) -> None:
    """
    Samples once from the replay buffer, and takes a single training step. The `step` argument is used to track the
    number of training steps taken.
    """
    data = self.buffer.sample(self.args.batch_size)  # o_t, a_t, r_{t+1}, d_{t+1}, o_{t+1}

    with t.inference_mode():
        target_max = self.target_network(data.next_obs).max(-1).values
    predicted_q_vals = self.q_network(data.obs).gather(-1, data.actions.unsqueeze(-1)).squeeze(-1)

    td_error = data.rewards + self.args.gamma * target_max * (1 - data.terminated.float()) - predicted_q_vals
    loss = td_error.pow(2).mean()
    loss.backward()
    self.optimizer.step()
    self.optimizer.zero_grad()

    if step % self.args.trains_per_target_update == 0:
        self.target_network.load_state_dict(self.q_network.state_dict())
```

</details>

<details>
<summary>Solution (full logging)</summary>

```python
def add_to_replay_buffer(self, n: int, verbose: bool = False):
    """
    Takes n steps with the agent, adding to the replay buffer. Should return a dict of data from the last
    terminated episode, if any (the training loop takes care of logging it).

    Optional argument `verbose`: if True, we can use a progress bar (useful to check how long the initial buffer
    filling is taking).
    """
    data = None

    for step in tqdm(range(n), disable=not verbose, desc="Adding to replay buffer"):
        infos = self.agent.play_step()

        # Get data from environments, if some environment did actually terminate
        new_data = get_episode_data_from_infos(infos)
        if new_data is not None:
            data = new_data  # makes sure we return a non-empty dict at the end, if some episode terminates

    return data

def prepopulate_replay_buffer(self):
    """
    Called to fill the replay buffer before training starts.
    """
    n_steps_to_fill_buffer = self.args.buffer_size // self.args.num_envs
    self.add_to_replay_buffer(n_steps_to_fill_buffer, verbose=True)

def training_step(self, step: int) -> None:
    """
    Samples once from the replay buffer, and takes a single training step. The `step` argument is used to track the
    number of training steps taken.
    """
    data = self.buffer.sample(self.args.batch_size)  # o_t, a_t, r_{t+1}, d_{t+1}, o_{t+1}

    with t.inference_mode():
        target_max = self.target_network(data.next_obs).max(-1).values
    predicted_q_vals = self.q_network(data.obs).gather(-1, data.actions.unsqueeze(-1)).squeeze(-1)

    td_error = data.rewards + self.args.gamma * target_max * (1 - data.terminated.float()) - predicted_q_vals
    loss = td_error.pow(2).mean()
    loss.backward()
    self.optimizer.step()
    self.optimizer.zero_grad()

    if step % self.args.trains_per_target_update == 0:
        self.target_network.load_state_dict(self.q_network.state_dict())

    # Log every 10th step
    if self.args.use_wandb and step % 10 == 0:
        wandb.log(
            {"td_loss": loss, "q_values": predicted_q_vals.mean().item(), "epsilon": self.agent.epsilon},
            step=self.agent.step,
        )
```

</details>

Here's some boilerplate code to test out your various probes, which you should make sure you're passing before testing on Cartpole.

In [41]:
def test_probe(probe_idx: int):
    """
    Tests a probe environment by training a network on it & verifying that the value functions are
    in the expected range.
    """
    # Train our network on this probe env.
    args = DQNArgs(
        env_id=f"Probe{probe_idx}-v0",
        wandb_project_name=f"test-probe-{probe_idx}",
        total_timesteps=3000 if probe_idx <= 2 else 5000,
        learning_rate=0.001,
        buffer_size=500,
        use_wandb=False,
        trains_per_target_update=20,
        video_log_freq=None,
        num_envs=1,
        steps_per_update=10,
        exploration_fraction=0.2,
        end_e=0.1,
    )
    trainer = DQNTrainer(args)
    trainer.train()

    # Get the correct set of observations, and corresponding values we expect
    obs_for_probes = [[[0.0]], [[-1.0], [+1.0]], [[0.0], [1.0]], [[0.0]], [[0.0], [1.0]]]
    expected_value_for_probes = [
        [[1.0]],
        [[-1.0], [+1.0]],
        [[args.gamma], [1.0]],
        [[-1.0, 1.0]],
        [[1.0, -1.0], [-1.0, 1.0]],
    ]
    tolerances = [5e-4, 5e-4, 5e-4, 5e-4, 1e-3]
    obs = t.tensor(obs_for_probes[probe_idx - 1]).to(device)

    # Calculate the actual value, and verify it
    value = trainer.q_network(obs)
    expected_value = t.tensor(expected_value_for_probes[probe_idx - 1]).to(device)
    t.testing.assert_close(value, expected_value, atol=tolerances[probe_idx - 1], rtol=0)
    print("Probe tests passed!\n")


for probe_idx in range(1, 6):
    test_probe(probe_idx)

  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/250 [00:00<?, ?it/s]

Probe tests passed!



  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/250 [00:00<?, ?it/s]

...

  0%|          | 0/450 [00:00<?, ?it/s]

Probe tests passed!



  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/450 [00:00<?, ?it/s]

Probe tests passed!



Once you've passed the tests for all 5 probe environments, you should test your model on Cartpole. We recommend you start by not using wandb until you can get it running without error, because this will improve your feedback loops (however if you've passed all probe environments then there's a good chance this code will just work for you).

You may find due to the small size of this network, and the few number of parallel environments, that
it may actually run faster on the CPU rather than the GPU. Try both and see.

In [42]:
args = DQNArgs(use_wandb=True)
trainer = DQNTrainer(args)
trainer.train()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 0/18125 [00:00<?, ?it/s]

SPS,▇▇▇▅▇▇▇▇▆▆▇█▆▇▇▆█▇▇█▇▅▇▇▇▇▆▆▅▆▇▇▆▁▇█▇▆▅▇
SPS,3058.45162


### Catastrophic forgetting

Note - you might see performance frequently drop off after it's achieved the maximum for a while, before eventually recovering again and repeating the cycle. Here's an example CartPole run using the solution code:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/cf3.png" width="550">

This is a well-known RL phenomenon called **catastrophic forgetting**. It happens when the replay buffer mostly contains successful experiences, and the model forgets how to adapt or recover from bad states. One way to fix this is to change your buffer to keep 10% of experiences from previous epochs, and 90% of experiences from the current phase. Can you implement this?

When we cover PPO, we'll also introduce **reward shaping**, which is another way this kind of behaviour can be mitigated.

#### Mechanism of catastrophic forgetting

Now that you have a working `DQNTrainer`, we can try to quantify catastrophic forgetting.
The cell below re-runs training with the default settings and, every 500 gradient steps,
records five things:

1. how long the greedy policy survives on fresh environments;
2. what fraction of the replay buffer's transitions are terminal (i.e. whether the data still contains any failures, which if we keep training on,
    the agent will forget what failure looks like)
3. the **action gap** $\max_a Q(s,a) - \min_a Q(s,a)$ averaged over a fixed batch of states
4. and the value $\max_a Q(s, a)$ the network assigns to a fixed set of **doomed states**: states from which *every* action ends the episode on the next step. By definition, doomed states $s$ satisfy $Q_{\pi}(s, a) = 0$ for any policy $\pi$
and any action $a$, as there is no reward, and no next state to bootstrap from.
5. the TD loss on a large sample from the buffer, the quantity `training_step` minimises.

Both fixed sets are sampled from the buffer at the start, while it is still full of
random-policy crashes.

In [44]:
# Re-run training with the default settings, but every 500 gradient steps record four things:
#   1. how long the greedy policy survives on 16 fresh environments (no exploration)
#   2. how many of the transitions in the replay buffer are terminal, i.e. does the data still contain failures
#   3. the action gap  max_a Q(s,a) - min_a Q(s,a)  averaged over a fixed batch of states from the random-policy phase
#   4. max_a Q(s, a) on a fixed set of doomed states, from which every action terminates next step (correct value 0)
#   5. the TD loss, recomputed on a large sample from the buffer (the same quantity training_step minimises)
from plotly.subplots import make_subplots
from rl_utils import greedy_rollout

# toggle this back to true to run this
RUN_ACTION_GAP_DEMO = True

if RUN_ACTION_GAP_DEMO:
    args = DQNArgs(use_wandb=False, steps_per_live_video=None)
    trainer = DQNTrainer(args)
    trainer.prepopulate_replay_buffer()
    probe_obs = trainer.buffer.sample(2048).obs  # sampled now, while the buffer is still full of random-policy failures
    pre_terminal_obs = trainer.buffer.obs[trainer.buffer.terminated]  # states whose taken action ended the episode
    # Check these are doomed under BOTH actions, by writing each state into a batched env and stepping it with each action.
    check_env = CartPole(num_envs=len(pre_terminal_obs), device=args.device)
    doomed = t.ones(len(pre_terminal_obs), dtype=t.bool, device=args.device)
    for a in (0, 1):
        check_env.state = pre_terminal_obs.clone()
        doomed &= check_env.step(t.full((len(pre_terminal_obs),), a, device=args.device))[2]  # terminated flag
    print(f"{doomed.sum().item()} of {len(doomed)} pre-termination states are doomed under both actions")
    doomed_obs = pre_terminal_obs[doomed]
    greedy = lambda obs: trainer.q_network(obs).argmax(-1)

    def td_loss(n_samples: int = 2048) -> float:
        d = trainer.buffer.sample(n_samples)
        with t.inference_mode():
            target = d.rewards + args.gamma * trainer.target_network(d.next_obs).max(-1).values * (1 - d.terminated.float())
            predicted = trainer.q_network(d.obs).gather(-1, d.actions.unsqueeze(-1)).squeeze(-1)
        return (target - predicted).pow(2).mean().item()

    log = []
    pbar = tqdm(range(args.total_training_steps))
    for step in pbar:
        trainer.add_to_replay_buffer(args.steps_per_update)
        trainer.training_step(step)
        if (step + 1) % 500 == 0:
            _, _, lifespan = greedy_rollout(CartPole(num_envs=16, seed=step, device=args.device), greedy)
            with t.inference_mode():
                q = trainer.q_network(probe_obs)
                q_doomed = trainer.q_network(doomed_obs).max(-1).values
            record = {
                "G" : lifespan.mean().item(),
                "n_term" : trainer.buffer.terminated.sum().item(),
                "act_gap" : (q.max(-1).values - q.min(-1).values).mean().item(),
                "q_doom" : q_doomed.mean().item(),
                "loss" : td_loss(),
            }
            log.append([step + 1, *record.values()])
            pbar.set_postfix({k: f"{v:.3g}" for k, v in record.items()})
else:
    print(f"{RUN_ACTION_GAP_DEMO=} Skipping action gap demo.")

  0%|          | 0/625 [00:00<?, ?it/s]

326 of 326 pre-termination states are doomed under both actions


  0%|          | 0/18125 [00:00<?, ?it/s]

In [46]:
if RUN_ACTION_GAP_DEMO:
    steps, greedy_lifespan, n_terminal, action_gap, q_doomed, td_loss_curve = zip(*log)
    fig = make_subplots(rows=5, cols=1, shared_xaxes=True, vertical_spacing=0.06, subplot_titles=[
        "Greedy episode length on 16 fresh envs",
        "Terminal transitions in the replay buffer (log scale; 0 shown at the bottom)",
        "Action gap  max_a Q(s,a) - min_a Q(s,a)  on a fixed batch of states (log scale)",
        "max_a Q(s, a) on fixed doomed states  (correct value 0)",
        "TD loss on 2048 sampled transitions (log scale)",
    ])
    n_terminal_plot = [max(n, 0.5) for n in n_terminal]  # a log axis can't show 0, so exact zeros sit at 0.5 with a tick labelled "0"
    for row, y in enumerate([greedy_lifespan, n_terminal_plot, action_gap, q_doomed, td_loss_curve], start=1):
        fig.add_scatter(x=steps, y=y, mode="lines+markers", showlegend=False, row=row, col=1)
    ticks = [0.5, 1, 10, 100, 1000]
    fig.update_yaxes(type="log", tickvals=ticks, ticktext=["0"] + [str(v) for v in ticks[1:]], range=[-0.4, 3], row=2, col=1)
    fig.update_yaxes(type="log", row=3, col=1)
    fig.update_yaxes(type="log", row=5, col=1)
    fig.update_layout(height=1100, width=800, xaxis5_title="gradient steps", margin=dict(t=40, b=40, l=40, r=20))
    fig.show()

<div style="text-align: left"><embed src="https://info-arena.github.io/ARENA_img/misc/media-22/2208.html" width="820" height="470"></div>

Panel by panel:

1. **The buffer runs out of failures.** 10,000 transitions across 16 envs is the last ~625 steps of play. Once every env survives all 500 steps, the buffer holds no terminal transitions (panel 2).
2. **Q stops distinguishing actions.** With no failures in the data, the TD targets say every action is fine, so $Q(s, \text{left})$ and $Q(s, \text{right})$ converge and the action gap decays (panel 3).
3. **Q forgets what doom looks like.** Doomed states are worth 0, but with nothing to anchor them their value drifts towards $1/(1-\gamma) = 100$, the value of never falling (panel 4). This begins while failures are merely rare, the impact to the loss is minimal, so they can be ignored.
4. **The loss can't see it.** Throughout the drift the TD loss is near zero (panel 5): with no failures in the buffer, $Q$ is perfectly consistent with the data it is trained on. Forgetting is invisible to the training signal.
5. **Collapse and recovery.** As the action gap approaches zero, the argmax policy becomes a coinflip,
the policy becomes random, and performance tanks (panel 1). This causes failures to re-enter the buffer, the loss spikes, and within a few hundred steps panels 3 and 4 snap back.

Panels 2-4 predict the actual collapse of the policy well in advance; the loss (panel 5) does not.
This motivates keeping old experiences, even if your current policy would never generate them,
as it reminds the Q-network not to forget.

If you let this run for an extened period (increase `total_timesteps`) you will see this cycle repeat.

# 4️⃣ Beyond CartPole & Bonus

## Beyond CartPole

If things go well and your agent masters CartPole, the next harder challenge is [MountainCar-v0](https://github.com/Farama-Foundation/Gymnasium/blob/main/gymnasium/envs/classic_control/mountain_car.py): `gpu_env.py` ships a batched GPU version of it too, "MountainCar-gpu"` (it has 3 discrete actions and a 2-dimensional observation, so `QNetwork` needs no changes). It's a much harder exploration problem than CartPole - the reward is -1 every step until the car reaches the flag, so random exploration almost never sees a success. Feel free to Google for appropriate hyperparameters - in a real RL problem you would have to do hyperparameter search using the techniques we learned on a previous day, because bad hyperparameters in RL often completely fail to learn even if the algorithm is perfectly correct.
There are many more exciting environments to play in, but generally they're going to require more compute and more optimization than we have time for today. If you finish the main material, some we recommend are:

- [Minimalistic Gridworld Environments](https://github.com/Farama-Foundation/gym-minigrid) - a fast gridworld environment for experiments with sparse rewards and natural language instruction.
- [microRTS](https://github.com/santiontanon/microrts) - a small real-time strategy game suitable for experimentation.
- [Megastep](https://andyljones.com/megastep/) - RL environment that runs fully on the GPU (fast!)
- [Procgen](https://github.com/openai/procgen) - A family of 16 procedurally generated gym environments to measure the ability for an agent to generalize. Optimized to run quickly on the CPU.
- [Atari](https://ale.farama.org/environments/) - although you might want to wait until the PPO section to try this on DQN, because we'll be going through some guided exercises implementing Atari with PPO there!

<details>
<summary>Some (very unpolished) code for setting up Atari with DQN</summary>

This is based on a hybrid of the agent/critic network setup for Atari, and the DQN implementation in this notebook. I've achieved decent performance in 40 mins training this, but not as good as we get when we do PPO on Atari in the PPO section, so I think this is somewhat underoptimized - if anyone finds improvements then feel free to make a PR!

It uses the same vectorised, GPU-tensor Atari environments as the PPO material (`rl_utils.AtariEnvs`, built on [EnvPool](https://envpool.readthedocs.io/)). Observations are the standard stack of 4 grayscale 84x84 frames, already scaled to $[0, 1]$, so the only thing you need to swap is the Q-network - the MLP below is replaced by the Nature-DQN convolutional net. Since `DQNTrainer` looks up the name `QNetwork` when it builds its networks, redefining the class here is all it takes. Two things to know: EnvPool runs with *episodic life* and *reward clipping* (so the progress bar's "episodes" are lives and rewards are $\pm 1$), and it resets an environment on the step *after* the one it ended on, so one terminal-observation-to-reset-observation transition per life ends up in the replay buffer - harmless here, but worth knowing about.

```python
def layer_init(layer: nn.Linear, std=np.sqrt(2), bias_const=0.0):
    t.nn.init.orthogonal_(layer.weight, std)
    t.nn.init.constant_(layer.bias, bias_const)
    return layer


class QNetwork(nn.Module):
    """The Nature-DQN convolutional Q-network, for (4, 84, 84) stacked-frame observations in [0, 1]."""

    layers: nn.Sequential

    def __init__(self, obs_shape: tuple[int, ...], num_actions: int, hidden_sizes: list[int] = [120, 84]):
        super().__init__()
        assert len(obs_shape) == 3, "Expecting (frames, height, width) observations - use the MLP QNetwork for CartPole"
        assert obs_shape[-1] % 8 == 4
        L_after_convolutions = (obs_shape[-1] // 8) - 3
        in_features = 64 * L_after_convolutions * L_after_convolutions

        self.layers = nn.Sequential(
            layer_init(nn.Conv2d(obs_shape[0], 32, 8, stride=4, padding=0)),
            nn.ReLU(),
            layer_init(nn.Conv2d(32, 64, 4, stride=2, padding=0)),
            nn.ReLU(),
            layer_init(nn.Conv2d(64, 64, 3, stride=1, padding=0)),
            nn.ReLU(),
            nn.Flatten(),
            layer_init(nn.Linear(in_features, 512)),
            nn.ReLU(),
            layer_init(nn.Linear(512, num_actions), std=0.01),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.layers(x)


# Register the environment: `ENVS` entries take (num_envs, seed, device); EnvPool picks its own device, so we ignore it
ENVS["ALE/Breakout-v5"] = lambda num_envs, seed, device: AtariEnvs("ALE/Breakout-v5", num_envs=num_envs, seed=seed)

args = DQNArgs(
    env_id="ALE/Breakout-v5",
    num_envs=4,
    buffer_size=1_000,  # 1000 x (4, 84, 84) float32 observations, twice (obs and next_obs) ~ 0.2 GB on the GPU
    batch_size=32,
    end_e=0.01,
    learning_rate=1e-4,
    total_timesteps=20_000,
    steps_per_update=5,
    steps_per_live_video=None,  # the 4x4 CartPole grid video doesn't apply here
    use_wandb=True,
    wandb_project_name="DQNAtari",
)
trainer = DQNTrainer(args)
trainer.train()
```

</details>

## Bonus

### Target Network

Why have the target network? Modify the DQN code above, but this time use the same network for both the target and the Q-value network, rather than updating the target every so often.

Compare the performance of this against using the target network.

### Shrink the Brain

Can DQN still learn to solve CartPole with a Q-network with fewer parameters? Could we get away with three-quarters or even half as many parameters? Try comparing the resulting training curves with a shrunken version of the Q-network. What about the same number of parameters, but with more/less layers, and less/more parameters per layer?

### Dueling DQN

Implement dueling DQN according to [the paper](https://arxiv.org/pdf/1511.06581.pdf) and compare its performance.